In [ ]:
def gene_dataset(mouseMine,KOBAS_identify):
    import pandas as pd
    from tqdm import tqdm
    from collections import defaultdict
    pd.options.mode.chained_assignment = None

    #Create the list of genes for the final table
    genes = pd.DataFrame(columns=['MGI_Gene_ID','Path','Phenotype'])
    genes.MGI_Gene_ID = mouseMine['Allele > Feature > Primary Identifier']
    genes = genes.drop_duplicates(ignore_index=True)

    #Creating a dictionary with genes as key and path id as values, used for faster comuputations
    df = KOBAS_identify[['ID','Input_MGI']]
    for i in range(0, len(df)):
        df.loc[i]['Input_MGI'] = (df.loc[i]['Input_MGI'].translate({ord(i): None for i in '[]\'\n'})).split(',')

    d = defaultdict(list)
    for idx, row in df.iterrows():
        gen = row['Input_MGI']
        for g in gen:
            d[g].append(row['ID'])

    #Creating the list of pathway for each gene (using the whole mousemine database takes up to 3/4 min)
    for i in tqdm(range(0,len(genes))):
        if (genes['MGI_Gene_ID'][i] in d.keys()):
            genes.Path[i] = d[genes['MGI_Gene_ID'][i]]
        genes.Phenotype[i] = mouseMine['Allele > Ontology Annotations > Ontology Term . Identifier'][mouseMine['Allele > Feature > Primary Identifier'] == genes.MGI_Gene_ID[i]].values

    genes = genes[genes['Phenotype'].notna()]
    genes.reset_index(inplace=True,drop=True)
    
    return(genes)